# 02 — Déployer un modèle, un Agent IA et un Workflow dans Microsoft Foundry

**Pré-requis :** avoir exécuté le notebook [01-fondamentaux-azure-paas.ipynb](01-fondamentaux-azure-paas.ipynb) et **gardé** le Resource Group + la ressource Foundry + le projet créés à la section 7.

**Objectif :** comprendre ce qu'est un **agent IA**, puis **dans le portail de la nouvelle Microsoft Foundry** (https://ai.azure.com, toggle *New Foundry* activé) :

1. Déployer un modèle (un LLM, ex. `gpt-5-mini`).
2. Le tester dans le **Chat playground**.
3. Créer un **prompt agent** simple avec un *system prompt* et un *function tool* custom.
4. Le tester dans l'**Agents playground**.
5. Créer un **workflow** (visual designer) qui orchestre **deux agents** en pattern *Sequential*.

Ce notebook est principalement **conceptuel et pas-à-pas dans le portail**. Le code Python pour appeler agents et workflows depuis une appli est documenté dans le [quickstart Foundry officiel](https://learn.microsoft.com/azure/foundry/quickstarts/get-started-code) (SDK `azure-ai-projects ≥ 2.0`).

> 🎯 À l'issue de ce notebook, vous saurez expliquer ce qu'est un agent IA, ses composants, les **3 types d'agents Foundry**, et vous aurez **votre propre agent + un workflow** opérationnels dans Foundry.
>
> ⚠️ Tout ce qui suit utilise la **nouvelle Foundry** (toggle *New Foundry* **activé** en haut du portail). L'expérience *classic* a une navigation et des chemins différents.

## 1. Qu'est-ce qu'un agent IA ?

Un **agent IA** n'est pas juste un LLM qui répond. C'est un LLM **placé dans une boucle de raisonnement**, à qui on a donné un **rôle**, des **outils** (= fonctions qu'il peut appeler) et une **mémoire**.

Là où un LLM brut vous répond à *une* question, un agent **décide quoi faire, agit, observe, et recommence** jusqu'à atteindre son objectif.

> 🪞 Analogie : un LLM brut = un stagiaire qui ne fait que répondre quand on lui parle. Un agent = un stagiaire qui sait aller chercher l'info dans l'intranet, appeler un collègue, faire un calcul, et vous revenir avec une réponse complète.

### 1.1 Anatomie d'un agent

```
┌────────────────────────────────────────────────────────────────────────┐
│                              AGENT IA                                  │
│                                                                        │
│   ┌────────────┐         ┌───────────────────────────────────────┐    │
│   │  INPUT     │ ──────► │  ORCHESTRATEUR / boucle de raisonnement│    │
│   │ (prompt    │         │                                       │    │
│   │  utilisat.)│         │   ┌─────────────────────────────────┐ │    │
│   └────────────┘         │   │  LLM (le « cerveau »)           │ │    │
│                          │   │  - system prompt (rôle)         │ │    │
│                          │   │  - historique de conversation   │ │    │
│                          │   │  - description des outils dispo │ │    │
│                          │   └────────┬────────────────────────┘ │    │
│                          │            │                          │    │
│                          │   ┌────────┴─────────┐                │    │
│                          │   ▼                  ▼                │    │
│                          │ appeler un       répondre à           │    │
│                          │ outil ?          l'utilisateur ?      │    │
│                          └────┬────────────────────┬─────────────┘    │
│                               │                    │                  │
│                               ▼                    ▼                  │
│                  ┌──────────────────────┐   ┌────────────┐            │
│                  │       OUTILS         │   │  OUTPUT    │            │
│                  │  (= fonctions que    │   │  (texte    │            │
│                  │   l'agent peut       │   │   final)   │            │
│                  │   invoquer)          │   └────────────┘            │
│                  │                      │                             │
│                  │  • web search        │                             │
│                  │  • calculator        │                             │
│                  │  • SQL query         │                             │
│                  │  • get_weather(city) │                             │
│                  │  • file_search       │                             │
│                  │  • code_interpreter  │                             │
│                  └──────────┬───────────┘                             │
│                             │ résultat                                │
│                             └──────────► (réinjecté dans le LLM)      │
│                                                                       │
│   ┌────────────────────────────────────────────────────────────────┐ │
│   │  MÉMOIRE                                                       │ │
│   │   • court terme : threads / historique du chat en cours        │ │
│   │   • long terme  : préférences, faits utilisateur persistants   │ │
│   │   • knowledge   : RAG sur documents indexés (vector store)     │ │
│   └────────────────────────────────────────────────────────────────┘ │
└────────────────────────────────────────────────────────────────────────┘
```

### 1.2 Les 5 composants en détail

| Composant | Rôle | Exemple |
|-----------|------|---------|
| **Input** | Le message de l'utilisateur (texte, image, audio, fichier…). | *« Combien de jours de congé me reste-t-il ? »* |
| **LLM** | Le cerveau. C'est lui qui *décide* quoi faire à chaque étape (appeler un outil ou répondre). Il est piloté par un **system prompt**. | GPT-4o, GPT-5, Llama 3, Phi-4… |
| **Outils** *(tools)* | Fonctions que l'agent peut appeler pour agir sur le monde extérieur. Le LLM **ne sait pas exécuter du code**, il **demande** à l'orchestrateur d'appeler tel outil avec tels arguments. | `get_remaining_leaves(employee_id)`, `search_internal_docs(query)`, `send_email(to, body)` |
| **Mémoire** | De quoi se souvient l'agent ? | court terme = thread de conversation ; long terme = profil utilisateur ; knowledge = documents RAG |
| **Output** | La réponse finale renvoyée à l'utilisateur (texte, citations, éventuellement actions). | *« Il vous reste 7 jours sur 2026, dont 3 doivent être posés avant fin juin. »* |

> 🧠 Le **system prompt** est crucial : c'est là qu'on définit le **rôle** (« Tu es un assistant RH… »), les **règles** (« Réponds toujours en français, cite tes sources, refuse les sujets hors RH ») et la **stratégie d'utilisation des outils** (« Pour toute question sur les congés, utilise `get_remaining_leaves` »).

### 1.3 Le cycle d'un tour de conversation

Quand l'utilisateur envoie un message, voici ce qui se passe (simplifié) :

```
 1. USER     : « Combien me reste-t-il de congés ? »
                       │
                       ▼
 2. LLM      : raisonne → « j'ai besoin de l'outil get_remaining_leaves »
                       │
                       ▼ (tool_call)
 3. TOOL     : get_remaining_leaves(employee_id=42) → { remaining: 7 }
                       │
                       ▼ (résultat réinjecté)
 4. LLM      : raisonne → « j'ai la donnée, je peux répondre »
                       │
                       ▼
 5. AGENT    : « Il vous reste 7 jours de congés. »
```

Le LLM peut faire **plusieurs tours** d'appels d'outils avant de répondre (`search → read → calculate → answer`). C'est la **boucle de raisonnement** ; on l'appelle aussi *ReAct* (Reason + Act).

### 1.4 Les 3 types d'agents dans la nouvelle Foundry

Le **Microsoft Foundry Agent Service** propose **trois types d'agents**, du plus simple au plus avancé. Comprendre cette distinction est important : on choisit le type selon le besoin.

| Type | Code requis ? | Hébergement | Orchestration | Idéal pour |
|------|---------------|-------------|---------------|------------|
| **Prompt agent** ⭐ | Non (portail) | Géré par Foundry | 1 seul agent | Prototypage rapide, outils internes, démos. **C'est ce qu'on va faire en §6.** |
| **Workflow agent** (preview) | Non (YAML optionnel) | Géré par Foundry | Multi-agents, branchements, human-in-the-loop | Processus métier multi-étapes, approbations. **On en fera un en §7.** |
| **Hosted agent** (preview) | Oui | Container sur Micro-VMs | Logique custom totale | Frameworks externes (Agent Framework, LangGraph), multi-agents complexes |

> 🎯 Pour ce notebook : on commence par un **prompt agent** (no-code), puis on l'orchestre dans un **workflow agent**. Les *hosted agents* sortent du périmètre de ce parcours.

### 1.5 Vocabulaire Foundry

Dans le **Foundry Agent Service**, on retrouve :

| Terme Foundry | Ce que c'est |
|---------------|--------------|
| **Model deployment** | Un modèle (ex. `gpt-5-mini`) déployé en endpoint dans votre projet, facturé au token. |
| **Agent** | La configuration : nom, modèle utilisé, *instructions* (system prompt), liste d'outils, knowledge attachée. Chaque modification = une nouvelle **version**. |
| **Tool** | Built-in (`file_search`, `code_interpreter`, `bing_grounding`, `azure_ai_search`, `openapi`, `function`, `mcp`, **A2A**…) ou **fonction custom**. |
| **Thread** | Un fil de conversation. Persistant côté Foundry, on peut le ré-ouvrir plus tard. |
| **Message** | Un tour dans le thread (rôle = user / assistant / tool). |
| **Run** | Une exécution de l'agent sur un thread = la boucle « raisonne → appelle outils → répond ». |
| **Workflow** | Une orchestration **déclarative** de plusieurs agents (Sequential, Group chat, Human-in-the-loop). |

✅ Avec ces concepts en tête, on passe à la **pratique dans le portail**.

## 2. Vérifier la ressource Foundry du notebook 01

On reprend simplement **le nom de votre Resource Group** (convention `rg-stage-<votre-identifiant>`, sur lequel vous êtes Owner). Tout le reste — l'identifiant, le nom de la ressource Foundry — est dérivé automatiquement.

In [ ]:
import os, re

# 👇 SEULE VARIABLE À ÉDITER : le nom de votre Resource Group (le même qu'au notebook 01).
#    Convention équipe : rg-stage-<votre-identifiant>
RG = "rg-stage-<votre-identifiant>"

# On dérive automatiquement l'identifiant et donc le nom de la ressource Foundry.
m = re.match(r"^rg-stage-(?P<id>[a-z0-9\-]+)$", RG)
assert m, f"❌ Le RG '{RG}' ne suit pas la convention 'rg-stage-<id>'."
USER_ID = m.group("id")

FOUNDRY  = f"aif-stage-{USER_ID}"
PROJECT  = "my-foundry-project"

for k, v in {"RG": RG, "USER_ID": USER_ID, "FOUNDRY": FOUNDRY, "PROJECT": PROJECT}.items():
    os.environ[k] = v

print(f"Resource Group    : {RG}")
print(f"Identifiant       : {USER_ID}")
print(f"Ressource Foundry : {FOUNDRY}")
print(f"Projet            : {PROJECT}")

In [ ]:
# Vérifier que la ressource Foundry et le projet existent toujours
!az cognitiveservices account show --name $FOUNDRY --resource-group $RG --query "{name:name, kind:kind, endpoint:properties.endpoint}" -o table
!az cognitiveservices account project show --name $FOUNDRY --resource-group $RG --project-name $PROJECT --query "{name:name, location:location}" -o table

## 3. Ouvrir le portail Microsoft Foundry

Tout ce qui suit se passe dans **https://ai.azure.com**.

### 3.1 Se connecter et sélectionner le projet

1. Allez sur **https://ai.azure.com** et connectez-vous avec le compte qui a créé la ressource au notebook 01.
2. **Vérifiez le toggle « New Foundry »** en haut de la page : il doit être **activé** (sinon vous êtes sur la version *classic* — les chemins ci-dessous ne s'appliquent pas).

   ![New Foundry toggle](https://learn.microsoft.com/azure/foundry/media/version-banner/new-foundry.png)

3. Le sélecteur de projet est en **haut à gauche**. Cliquez dessus et choisissez `my-foundry-project` (celui créé en 01).
4. Si vous ne le voyez pas : changez de tenant/abonnement via le sélecteur en haut à droite.

### 3.2 La navigation de la nouvelle Foundry

Dans la **nouvelle Foundry**, le menu principal est en haut à droite et propose deux sections :

- **Build** → c'est ici qu'on développe : **Models + endpoints**, **Playgrounds** (Chat, Agents…), **Agents**, **Knowledge**, **Workflows**.
- **Operate** → administration : **Admin**, projets, accès, quotas.

> 💡 La plupart de ce notebook se fait depuis **Build**. C'est différent de la Foundry « classic » qui avait des hubs et un menu de gauche très différent.

> 🔐 Si l'accès est refusé : il faut avoir un rôle **Foundry User** (anciennement *Azure AI User*) ou supérieur sur le projet. Voir https://learn.microsoft.com/azure/foundry/concepts/rbac-foundry

## 4. Déployer un modèle

Un modèle dans le **catalogue** est juste « disponible », il ne tourne pas encore. Pour pouvoir l'appeler, il faut le **déployer** : Azure provisionne un endpoint, et vous serez facturé à l'usage (au token).

### 4.1 Choisir un modèle

Pour démarrer, on prend un modèle **petit, rapide et peu cher** : **`gpt-5-mini`** (alternative : `gpt-4.1-mini` ou `gpt-4o-mini`).

| Modèle | Famille | Quand l'utiliser |
|--------|---------|------------------|
| **`gpt-5-mini`** ⭐ | OpenAI petit, multimodal | Chat simple, latence faible, coût bas — **notre choix par défaut** |
| `gpt-4.1-mini` / `gpt-4o-mini` | OpenAI petit (générations précédentes) | Compatibilité, encore largement déployable |
| `gpt-5` / `gpt-4.1` | OpenAI grand | Raisonnement complexe, agents multi-étapes |
| `Phi-4-mini` | SLM Microsoft | Edge, on-device, très petit budget |
| `Llama-3.3-70B-Instruct` | Meta, grand | Open-weight, alternative à OpenAI |
| `text-embedding-3-large` | Embeddings | Pour faire du RAG (notebooks suivants) |

### 4.2 Pas-à-pas dans le portail

1. Dans votre projet : menu **`Build`** (en haut à droite) → **`Models + endpoints`**.
2. Cliquez **`Deploy model`** → **`Deploy base model`**.
3. Dans la barre de recherche, tapez **`gpt-5-mini`** (ou `gpt-4.1-mini`). Sélectionnez-le, puis **`Confirm`**.
4. Une fenêtre **Deploy model** s'ouvre :
   - **Deployment name** : laissez la valeur par défaut (ex. `gpt-5-mini`) — c'est ce nom qu'on utilisera dans le code et l'agent. **Notez-le.**
   - **Deployment type** : **`Global Standard`** (le plus simple, dispo partout).
   - **Model version** : la dernière proposée.
   - **Tokens per minute rate limit (TPM)** : laissez la valeur par défaut.
5. Cliquez **`Deploy`**. Attendez ~30 secondes.
6. Vous voyez votre déploiement dans la liste **Models + endpoints**, avec son **endpoint URL** et son **deployment name**.

> ⚡ **Raccourci** : depuis la **page d'accueil** de Foundry, le bouton **`Start building → Create agent`** crée en un seul geste un projet (si vous n'en aviez pas), déploie un modèle par défaut, et crée un agent. Pratique pour démarrer de zéro — mais comme on a déjà un projet du notebook 01, on passe par `Build → Models + endpoints`.

> 💡 La région du déploiement doit héberger ce modèle. Si la création échoue, essayez un autre modèle ou changez la région de la ressource Foundry.

### 4.3 Vérification en ligne de commande (optionnel)

On peut aussi lister les déploiements via l'`az cli` pour vérifier ce que vient de faire le portail.

In [ ]:
!az cognitiveservices account deployment list \
    --name $FOUNDRY --resource-group $RG \
    --query "[].{name:name, model:properties.model.name, version:properties.model.version, sku:sku.name, capacity:sku.capacity}" \
    --output table

## 5. Tester le modèle dans le Chat playground

Le playground vous permet de **parler au modèle directement**, sans agent ni outil. C'est utile pour :

- valider qu'un déploiement répond bien,
- prototyper un *system prompt*,
- comparer plusieurs modèles côte à côte.

### 5.1 Ouvrir le Chat playground

1. Menu **`Build`** → **`Playgrounds`** → **`Chat`** (la tuile *Chat playground*).
2. En haut, **`Deployment`** = sélectionnez votre `gpt-5-mini`.
3. À gauche, **`Setup` → `Give the model instructions and context`** : c'est le **system prompt**.

### 5.2 Essais à faire

**Essai 1 — sans system prompt :**
Tapez : *« Présente-toi en deux phrases. »*  
Observation : le modèle se présente comme un assistant générique.

**Essai 2 — avec un system prompt :** dans le bloc Setup, collez :
```
Tu es Léo, un assistant pédagogique pour stagiaires qui découvrent Azure.
Réponds toujours en français, en deux phrases maximum, avec un ton bienveillant.
Termine chaque réponse par un emoji adapté.
```
Appliquez (`Apply`), puis tapez la même question. Le modèle se présente cette fois comme **Léo**, court et avec un emoji. **C'est la puissance du system prompt.**

**Essai 3 — paramètres :** à droite, ajustez :
- **Temperature** : 0 = déterministe et factuel, 1 = créatif. Essayez `0.2` puis `1.0` sur *« Donne-moi un slogan pour une marque de chaussettes. »*
- **Max response tokens** : la longueur max de la réponse.
- **Top P / Frequency penalty / Presence penalty** : pour plus tard.

> 📊 L'onglet **`Tokens`** (ou panneau « Usage ») en haut du playground vous montre combien de tokens d'entrée et de sortie ont été consommés. Très instructif pour comprendre la facturation.

## 6. Créer un **Prompt agent**

Maintenant qu'on a un modèle qui répond, on monte d'un cran : on crée un **prompt agent** — c'est-à-dire un modèle **+ des instructions persistantes + des outils + (optionnel) de la knowledge**.

### 6.1 Créer l'agent

1. Menu **`Build`** → **`Agents`** (panneau de gauche).
2. Cliquez **`+ Create agent`** (ou **`+ New agent`** selon la version).
3. Donnez-lui un nom : `assistant-stagiaire`, une description courte, puis **`Create`**.
4. Vous arrivez sur la **page de l'agent** (parfois appelée *Agent playground*).
5. Configurez les champs principaux :
   - **Deployment** (ou *Model*) : votre `gpt-5-mini`.
   - **Instructions** (= system prompt persistant) : collez le bloc ci-dessous.

```
Tu es Léo, un assistant pédagogique pour des stagiaires qui apprennent Azure.

Règles :
- Réponds toujours en français, clair et concis.
- Quand l'utilisateur pose une question sur l'heure ou la date, appelle l'outil `get_current_time`
  au lieu d'inventer.
- Cite tes sources si tu utilises un outil de recherche.
- Refuse poliment toute question qui n'est pas liée à l'apprentissage du cloud / de l'IA.

Termine chaque réponse par un emoji adapté.
```

6. Cliquez **`Save`** (très important : Foundry **ne sauvegarde pas automatiquement**).

> 🆕 Chaque `Save` crée une **nouvelle version immuable** de l'agent. Vous pouvez revenir à une version antérieure si besoin.

### 6.2 Ajouter un outil custom (`function`)

On va donner à l'agent **un outil très simple** : `get_current_time`. C'est un **function tool** : l'agent **demande** à l'appeler avec des arguments, et c'est **votre code** qui exécute et renvoie la réponse (dans le playground, on simule la réponse à la main).

1. Sur la page de l'agent, section **`Tools`** → **`+ Add`** → onglet **`Custom`** → **`Function`** (autres choix possibles dans cet onglet : `OpenAPI tool`, `MCP server`, etc.).
2. Collez ce **schéma JSON** (format function-calling standard) :

```json
{
  "name": "get_current_time",
  "description": "Retourne l'heure et la date actuelles dans le fuseau demandé (par défaut Europe/Paris).",
  "parameters": {
    "type": "object",
    "properties": {
      "timezone": {
        "type": "string",
        "description": "IANA timezone, par exemple Europe/Paris, America/New_York, Asia/Tokyo.",
        "default": "Europe/Paris"
      }
    },
    "required": []
  }
}
```

3. **`Create`** puis **`Save`** sur la page de l'agent pour persister la nouvelle version.

> 🧩 L'onglet **`Catalog`** (à côté de `Custom`) propose aussi des outils prêts à brancher : **Bing grounding**, **Azure AI Search**, **Azure Speech MCP Server**, **Microsoft Fabric**, etc. On en utilise un en bonus à la section 8.

> 💡 Dans la vraie vie, l'implémentation de `get_current_time` est dans votre code (via le SDK `azure-ai-projects`, cf. [quickstart officiel](https://learn.microsoft.com/azure/foundry/quickstarts/get-started-code)). Dans le playground, Foundry vous **demande la réponse de l'outil à la main**.

### 6.3 Tester l'agent dans le playground

Sur la page de l'agent, cliquez **`Try in playground`** (panneau de droite ou bouton en haut).

**Essai 1 — interaction simple (pas d'outil)**

> 👤 *« Salut, qui es-tu ? »*

Réponse attendue : Léo se présente brièvement, en français, avec un emoji.

**Essai 2 — déclencher l'outil**

> 👤 *« Quelle heure est-il à Tokyo ? »*

Vous devriez voir l'agent **appeler `get_current_time`** avec `{ "timezone": "Asia/Tokyo" }`. Comme c'est une *function tool* (pas un built-in), le playground vous **demande de fournir la réponse de l'outil à la main**. Tapez par exemple :

```json
{ "datetime": "2026-05-25T22:14:00+09:00", "timezone": "Asia/Tokyo" }
```

Validez. L'agent reprend la main, lit le résultat, et formule la réponse finale à l'utilisateur.

**Essai 3 — hors-sujet**

> 👤 *« Donne-moi la recette du gâteau au chocolat. »*

L'agent doit **refuser poliment** (c'est dans son system prompt).

### 6.4 Inspecter le run

Dans le playground, cliquez sur un message → **`Show details`** / **`Thread logs`** : vous voyez les étapes du **run** — appel d'outil, arguments, réponse, raisonnement intermédiaire. **C'est exactement ce qu'on va monitorer via Application Insights dans le notebook 06.**

## 7. Créer un **Workflow agent** — orchestrer plusieurs agents

Un **prompt agent** = 1 LLM dans une boucle.  
Un **workflow agent** = **plusieurs agents** orchestrés dans un graphe **visuel**, avec branchements, variables et étapes humaines optionnelles. Pas de code, le résultat est sauvegardé en YAML versionné côté Foundry.

> 🚧 Les workflow agents sont en **public preview** au moment où ce notebook est écrit. La doc de référence : https://learn.microsoft.com/azure/foundry/agents/concepts/workflow

### 7.1 Quand utiliser un workflow ?

Pensez workflow quand :

- vous voulez **enchaîner plusieurs agents spécialisés** (ex. *extracteur → traducteur → résumeur*) ;
- vous avez besoin de **branchements** (`if/else`) ou de variables sans écrire de code ;
- vous voulez une **étape humaine** (validation, clarification) au milieu du process.

Sinon, un simple prompt agent suffit.

### 7.2 Les 3 patterns de workflow

Foundry fournit des **templates** pour les patterns d'orchestration les plus courants :

| Pattern | Description | Cas d'usage typique |
|---------|-------------|---------------------|
| **Sequential** | Le résultat d'un agent est passé au suivant, dans un ordre défini. | Pipelines en étapes : extraction → transformation → résumé |
| **Group chat** | Le contrôle passe **dynamiquement** entre agents selon le contexte. | Escalade, fallback, expert handoff |
| **Human-in-the-loop** | Le workflow s'arrête, pose une question à l'utilisateur, et attend sa réponse. | Approbations, demandes de précision |

### 7.3 Pas-à-pas — créer un workflow Sequential

Objectif pédagogique : un mini-pipeline à **2 agents** — `traducteur-fr-en` puis `resumeur-en`. L'utilisateur entre un texte en français, le workflow le traduit en anglais, puis le résume.

#### Étape 1 — Préparer les agents (réutilisation)

On va d'abord créer (ou réutiliser) deux prompt agents simples. Refaites rapidement la **section 6.1** deux fois :

1. **Agent 1** : nom `traducteur-fr-en`, deployment `gpt-5-mini`, instructions :
   ```
   Tu es un traducteur professionnel français → anglais.
   Traduis fidèlement le texte fourni en anglais, sans rien ajouter.
   Ne réponds QUE par la traduction, pas de préambule.
   ```
   `Save`.

2. **Agent 2** : nom `resumeur-en`, deployment `gpt-5-mini`, instructions :
   ```
   You are a concise summarizer.
   Summarize the input text in 2 short sentences in English.
   Output only the summary, no preamble.
   ```
   `Save`.

#### Étape 2 — Créer le workflow

1. Menu **`Build`** → **`Create new workflow`** → choisissez le template **`Sequential`**.
2. Foundry ouvre le **visualizer** : une chaîne de **nœuds** (Trigger → Agent → Agent → Output).
3. Cliquez sur le **premier nœud agent** : 
   - **`Invoke agent`** → onglet **`existing`** → tapez `traducteur-fr-en` → sélectionnez.
4. Cliquez sur le **deuxième nœud agent** : sélectionnez `resumeur-en`.
5. ⚠️ **Cliquez `Save` en haut du visualizer.** Foundry **ne sauvegarde rien automatiquement**, c'est la règle d'or des workflows.

#### Étape 3 — Tester le workflow

1. Toujours dans le visualizer : cliquez **`Run Workflow`** (en haut).
2. Une fenêtre de chat s'ouvre. Tapez par exemple :
   > *Le cloud Azure permet de déployer rapidement des applications sans gérer de serveurs physiques.*
3. Vous voyez **chaque nœud s'allumer** au fur et à mesure :
   - `traducteur-fr-en` retourne *« Azure cloud lets you quickly deploy applications without managing physical servers. »*
   - `resumeur-en` retourne *« Azure enables fast, server-free app deployment. »*
4. La sortie finale apparaît dans la fenêtre de chat.

### 7.4 Ajouter du contrôle de flux

Pour aller un peu plus loin, vous pouvez ajouter via le bouton **`+`** entre deux nœuds :

| Bloc | Rôle |
|------|------|
| **Set variable** | Stocker une valeur intermédiaire dans le workflow. |
| **Condition** | Branchement `if/else` sur une variable ou la sortie d'un agent. |
| **Question** | **Human-in-the-loop** : pose une question à l'utilisateur et met en pause. |
| **Invoke agent** | Ajouter un agent supplémentaire (existant ou nouveau, créé inline). |
| **End** | Terminer explicitement le workflow. |

Pour structurer la sortie d'un agent, utilisez **`Details → Text format → JSON Schema`** : l'agent renverra du JSON conforme au schéma fourni, ce qui rend les variables aval beaucoup plus fiables.

> 🚫 Note : **les hosted agents ne sont PAS supportés** comme nœuds dans le workflow designer. Si vous avez besoin d'orchestration custom autour d'un hosted agent, il faut le faire dans le code de l'hosted agent (via Microsoft Agent Framework).

### 7.5 Versionnage & supervision

- Chaque `Save` crée une **nouvelle version immuable** du workflow.
- L'historique est visible dans la page du workflow → onglet **`Versions`**.
- Les runs et leurs traces atterrissent dans Application Insights (notebook 06) si vous avez connecté l'observabilité.

## 8. (Bonus) Ajouter de la knowledge — RAG en un clic

Si vous voulez aller plus loin maintenant, vous pouvez attacher de la **knowledge** à votre prompt agent : il pourra alors répondre à partir de **vos documents**, avec citations.

1. Sur la page de l'agent → section **`Knowledge`** → **`+ Add`**.
2. Choisissez **`Files`** (le plus simple) → uploadez 2-3 PDF / Markdown sur un sujet (ex. doc interne, FAQ).
3. Foundry crée automatiquement un **vector store**, fait les **embeddings**, et rattache un outil **`file_search`** à votre agent.
4. **`Save`** sur l'agent.
5. Re-testez dans le playground : *« Que dit le document X sur le sujet Y ? »*  
   L'agent appelle `file_search`, retrouve les passages pertinents et répond avec **citations**.

C'est le mécanisme **RAG** qu'on a vu en 3.1.5 du notebook 01, ici sans écrire une ligne de code.

> 💡 Pour des knowledge bases plus avancées (multi-sources, partagées entre agents), regardez **Foundry IQ** : https://learn.microsoft.com/azure/foundry/agents/concepts/what-is-foundry-iq

## 9. Récupérer l'endpoint du projet pour le SDK

Pour appeler agents et workflows depuis du code (SDK `azure-ai-projects ≥ 2.0`), vous aurez besoin du **project endpoint**. On le récupère via l'`az cli` ou directement dans le portail (sur la page d'accueil du projet, encadré **Project endpoint**).

In [ ]:
# Récupérer l'endpoint Foundry (ressource) puis construire l'endpoint du projet
import subprocess
FOUNDRY_ENDPOINT = subprocess.check_output(
    f'az cognitiveservices account show --name {FOUNDRY} --resource-group {RG} --query properties.endpoint -o tsv',
    shell=True,
).decode().strip()

# Format attendu par le SDK azure-ai-projects >= 2.0 :
#   https://<foundry>.services.ai.azure.com/api/projects/<project>
PROJECT_ENDPOINT = f"{FOUNDRY_ENDPOINT.rstrip('/')}/api/projects/{PROJECT}"

print(f"Endpoint ressource Foundry : {FOUNDRY_ENDPOINT}")
print(f"Endpoint du projet (SDK)   : {PROJECT_ENDPOINT}")

> 💡 Les opérations sur les **agents** (list, get, run) se font côté **data plane** via le SDK **`azure-ai-projects` ≥ 2.0** (déjà installé par le `requirements.txt` du notebook 01) — exemple :
> ```python
> from azure.identity import DefaultAzureCredential
> from azure.ai.projects import AIProjectClient
> project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential())
> # project.agents.list() / project.agents.create_version(...) / etc.
> ```
> L'`az cli` couvre surtout le **management plane** (créer ressources, déploiements, projets). Pour les appels SDK complets, voir le [quickstart Foundry](https://learn.microsoft.com/azure/foundry/quickstarts/get-started-code).

## 10. 🧹 Nettoyage

Le **playground**, les **agents** et les **workflows** ne coûtent rien tant que vous n'envoyez pas de messages.  
Le **déploiement du modèle** est facturé **au token** — pas à l'heure. Vous pouvez donc le laisser sans frais récurrents tant que personne ne l'appelle.

Si vous voulez quand même supprimer le déploiement maintenant :

In [ ]:
# Lister puis (optionnel) supprimer un déploiement spécifique.
DEPLOYMENT_NAME = "gpt-5-mini"  # adaptez si vous l'avez nommé autrement

!az cognitiveservices account deployment list --name $FOUNDRY --resource-group $RG --query "[].name" -o table

# Décommentez pour supprimer :
# !az cognitiveservices account deployment delete --name $FOUNDRY --resource-group $RG --deployment-name {DEPLOYMENT_NAME}

> ⚠️ **Ne supprimez pas le Resource Group** : il appartient à votre équipe et peut contenir d'autres ressources. Faites du nettoyage **ressource par ressource** (cf. cellule ci-dessus pour le déploiement de modèle ; pour la Web App et Foundry, voir le notebook 01).
>
> 💡 Si vous comptez enchaîner sur les notebooks **03 (identité)**, **05 (appel SDK)** ou **06 (monitoring + évaluation)**, **gardez** tout en place.

## Récap

Vous savez maintenant :
- expliquer ce qu'est un **agent IA** : input, LLM, **outils**, **mémoire**, output, et la **boucle de raisonnement** ;
- la différence entre **LLM brut** et **agent** ;
- les **3 types d'agents Foundry** : **prompt** (no-code), **workflow** (orchestration visuelle), **hosted** (container) ;
- le vocabulaire Foundry : *deployment, agent, version, thread, message, run, tool, workflow* ;
- **déployer un modèle** dans Foundry depuis le portail (menu **Build**) ;
- **tester un modèle** dans le **Chat playground** (system prompt, temperature, tokens) ;
- **créer un prompt agent** avec instructions + un outil custom (function-calling) ;
- **tester un agent** dans l'**Agents playground**, et inspecter un *run* ;
- **créer un workflow Sequential** orchestrant deux agents ;
- récupérer l'endpoint du projet pour le SDK Python.

**Prochaines étapes du parcours :**
- Notebook 03 — **Projet, DevOps & Agile** : comment ce qu'on vient de faire s'inscrit dans un projet en équipe.
- Notebook 04 — **Architecture web** : où s'insère un agent IA dans une appli web réelle.
- Notebook 05 — **Sécurité cloud** : authentifier votre code à Foundry **sans clé** avec Managed Identity.
- Notebook 06 — **Monitoring & évaluation** : voir les *runs*, traces, tokens, et mesurer la qualité.

📚 Pour aller plus loin :
- Vue d'ensemble Foundry Agent Service : https://learn.microsoft.com/azure/foundry/agents/overview
- Quickstart prompt agent (portail + SDK) : https://learn.microsoft.com/azure/foundry/quickstarts/get-started-code
- Workflows : https://learn.microsoft.com/azure/foundry/agents/concepts/workflow
- Hosted agents (preview) : https://learn.microsoft.com/azure/foundry/agents/concepts/hosted-agents
- Catalogue des outils (built-in & MCP) : https://learn.microsoft.com/azure/foundry/agents/how-to/tools/overview
- Catalogue de modèles : https://learn.microsoft.com/azure/foundry/model-catalog-overview